# Introducción

Este notebook presenta la implementación y evaluación de diferentes arquitecturas de Deep Learning utilizando TensorFlow y Keras. Se desarrollan modelos de redes neuronales densas para tareas de regresión y redes neuronales convolucionales para clasificación de imágenes. Además, se analizan distintas técnicas de regularización y estrategias de entrenamiento para mejorar el rendimiento de los modelos.

In [ ]:
# Imports
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
import random
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)
keras.utils.set_random_seed(0)

# Redes Densas

Vamos a utilizar el [California Housing Dataset](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) para predecir el precio mediano de las viviendas en diferentes distritos de California.

El precio de las viviendas es un valor continuo que está expresado en cientos de miles de dólares (por ejemplo, un valor de 2.5 significa $250,000). Por lo tanto, el problema es una `regresión`.

In [ ]:
# Cargar el dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# Mostrar información básica
print(f"Tamaño del dataset: {df.shape}")
print(f"Rango de precios: ${df['MedHouseVal'].min():.1f}k - ${df['MedHouseVal'].max():.1f}k")
print("\nPrimeras 5 filas:")
df.head()

In [ ]:
# Separar features y target
y = df.pop('MedHouseVal').values
X = df.copy().values

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

print('x_train, y_train shapes:', x_train.shape, y_train.shape)
print('x_test, y_test shapes:', x_test.shape, y_test.shape)
print('Some prices: ', y_train[:5])

In [ ]:
# Normalizar las features
normalizer = layers.Normalization()

normalizer.adapt(x_train)

x_train = normalizer(x_train)
x_test = normalizer(x_test)


## Modelo base de red neuronal densa

In [ ]:
# Modelo
model = tf.keras.models.Sequential([
    layers.Input(shape=(x_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

In [ ]:
# Compilación del modelo
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [ ]:
model.fit(x_train,
          y_train,
          epochs=60,
          batch_size=32,
          validation_split=0.2,
          verbose=1)

In [ ]:
results = model.evaluate(x_test, y_test, verbose=1)
test_metrics = dict(zip(model.metrics_names, results))
print('Test Metrics:', test_metrics)

## Mejora del modelo mediante regularización

In [ ]:
# Modelo
model = tf.keras.models.Sequential([

    layers.Input(shape=(x_train.shape[1],)),

    layers.Dense(
        128,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),
    layers.Dropout(0.3),

    layers.Dense(
        128,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),
    layers.Dropout(0.3),

    layers.Dense(
        64,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),

    layers.Dense(
        64,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),

    layers.Dense(1)
])

In [ ]:
# Compilación del modelo
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [ ]:
batch_size = 32

In [ ]:
model.fit(x_train,
          y_train,
          epochs=60,
          batch_size=batch_size,
          validation_split=0.2,
          verbose=1)

In [ ]:
results = model.evaluate(x_test, y_test, verbose=1)
test_metrics = dict(zip(model.metrics_names, results))
print('Test Metrics:', test_metrics)

## Prevención del sobreajuste mediante Early Stopping

In [ ]:
# Modelo
model = tf.keras.models.Sequential([

    layers.Input(shape=(x_train.shape[1],)),

    layers.Dense(
        128,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),
    layers.Dropout(0.3),

    layers.Dense(
        128,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),
    layers.Dropout(0.3),

    layers.Dense(
        64,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),

    layers.Dense(
        64,
        activation='relu',
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),

    layers.Dense(1)
])

In [ ]:
# Compilación del modelo
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [ ]:
# Definir el early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model.fit(
    x_train,
    y_train,
    epochs=60,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[early_stopping]
)

In [ ]:
results = model.evaluate(x_test, y_test, verbose=1)
test_metrics = dict(zip(model.metrics_names, results))
print('Test Metrics:', test_metrics)

# Redes Convolucionales

Vamos a usar el dataset [cifar-10](https://www.cs.toronto.edu/~kriz/cifar.html), que son 60000 imágenes de 32x32 a color  con 10 clases diferentes.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i])
    plt.xlabel(class_names[y_train[i]])
plt.show()

In [ ]:
print('x_train, y_train shapes:', x_train.shape, y_train.shape)
print('x_test, y_test shapes:', x_test.shape, y_test.shape)

## Red convolucional utilizando la API Funcional

In [ ]:
inputs = tf.keras.Input(shape=(32, 32, 3), name='input')

# Rescaling
x = layers.Rescaling(1./255)(inputs)

# Bloque 1
x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)

# Bloque 2
x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)

# Bloque 3
x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)

# Flatten
x = layers.Flatten()(x)

# Fully connected
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(10, activation='softmax')(x)

model = keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
# Compilación del modelo
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Definir un callback para early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=25,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stopping]
)

In [ ]:
results = model.evaluate(x_test, y_test, verbose=1)

test_metrics = dict(zip(model.metrics_names, results))
for metric in model.metrics_names:
    print(f"test_{metric}: {test_metrics[metric]}")

## Implementación de una CNN con la API Secuencial

In [ ]:
# Modelo
model_seq = tf.keras.models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])